# Game of Thrones dataset

This notebook loads a small Game of Thrones-style graph into both
`NetworkXGraph` and `Neo4jGraph`.

Dataset notes:
- character nodes built from a public Thrones API
- house nodes linked with `MEMBER_OF`
- bundled fallback data keeps the example runnable offline

In [ ]:
import importlib.util

# Comprovació de presència del paquet
package_to_check = 'drm'
spec = importlib.util.find_spec(package_to_check)

if spec is None:
    print(f'⚠️ {package_to_check} no està instal·lat. Iniciant instal·lació...')
    %pip install -q --upgrade drm-tools
    print("✅ Instal·lació completada. L'estat del kernel PODRIA requerir un reinici.")
else:
    print(f'✅ {package_to_check} ja està present al sistema. Saltant instal·lació.')


In [ ]:
import os

from drm import Neo4jGraph, NetworkXGraph
from drm.exemples import load_got_characters


def neo4j_config(default_target="DEV"):
    target = os.environ.get("NEO4J_TARGET", default_target).upper()
    prefix = f"NEO4J_{target}_"
    return {
        "target": target,
        "url": os.environ.get(f"{prefix}URL", os.environ.get("NEO4J_URL", "bolt://localhost:7687")),
        "user": os.environ.get(f"{prefix}USER", os.environ.get("NEO4J_USER", "neo4j")),
        "password": os.environ.get(f"{prefix}PASSWORD", os.environ.get("NEO4J_PASSWORD", "secret")),
        "database": os.environ.get(f"{prefix}DATABASE", os.environ.get("NEO4J_DATABASE", "neo4j")),
    }

## NetworkXGraph

In [ ]:
nx_graph = NetworkXGraph()
nx_stats = load_got_characters(nx_graph, limit=25)
print("NetworkX stats:", nx_stats)
nx_graph.close()

## Neo4jGraph

In [ ]:
cfg = neo4j_config()
print("Using Neo4j target:", cfg["target"])
neo4j_graph = Neo4jGraph(
    url=cfg["url"],
    user=cfg["user"],
    password=cfg["password"],
    database=cfg["database"],
)
try:
    neo4j_stats = load_got_characters(neo4j_graph, limit=25)
    print("Neo4j stats:", neo4j_stats)
finally:
    neo4j_graph.close()
